In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_ground_truth.tsv
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source3.tsv
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source2.tsv
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source1.tsv


In [9]:
import os

print("INPUT DIRECTORIES:")
for x in os.listdir("/kaggle/input"):
    print(repr(x))

print("\nFULL TREE:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

INPUT DIRECTORIES:
'datasets'

FULL TREE:
input/
  datasets/
    fuckinggpu/
      amazon-entity-resolution-data/
        train/
          train_ground_truth.tsv
          train_source3.tsv
          train_source2.tsv
          train_source1.tsv


In [10]:
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data"
)

for p in sorted(DATA_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p}  |  {p.stat().st_size / 1024**2:.1f} MB")

/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_ground_truth.tsv  |  121.1 MB
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source1.tsv  |  200.3 MB
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source2.tsv  |  466.6 MB
/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train/train_source3.tsv  |  480.4 MB


In [ ]:
!pip install -q "rapidfuzz>=3.6"

In [11]:
import sys
import pandas
import numpy
import scipy
import sklearn
import lightgbm
import rapidfuzz
import pyarrow
import joblib
import psutil

print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)
print("rapidfuzz:", rapidfuzz.__version__)
print("pyarrow:", pyarrow.__version__)
print("joblib:", joblib.__version__)
print("psutil:", psutil.__version__)

ModuleNotFoundError: No module named 'rapidfuzz'

In [13]:
!pip install "rapidfuzz>=3.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.6 MB/s eta 0:00:00:00:01


In [14]:
import rapidfuzz
print("RapidFuzz:", rapidfuzz.__version__)

RapidFuzz: 3.14.6


In [16]:
from pathlib import Path
import shutil

SRC = Path("/kaggle/input/datasets/fuckinggpu/business-entity-resolution-source")
if not SRC.exists():
    SRC = Path("/kaggle/input/business-entity-resolution-source")
WORK = Path("/kaggle/working/ml_amazon")

# Remove only the partially-created code directory
if (WORK / "code").exists():
    shutil.rmtree(WORK / "code")

(WORK / "code").mkdir(parents=True, exist_ok=True)

if (SRC / "code").exists():
    shutil.copytree(SRC / "code", WORK / "code", dirs_exist_ok=True)
elif (SRC / "business_entity_resolution").exists():
    shutil.copytree(SRC / "business_entity_resolution", WORK / "code" / "business_entity_resolution", dirs_exist_ok=True)
else:
    for p in SRC.glob("*"):
        if p.is_dir() and "business_entity_resolution" in p.name:
            shutil.copytree(p, WORK / "code" / "business_entity_resolution", dirs_exist_ok=True)
            break
        elif (p / "src").exists():
            shutil.copytree(p, WORK / "code" / p.name, dirs_exist_ok=True)

print("Source copied successfully.")
print("\nPython files:")

for p in sorted((WORK / "code").rglob("*.py")):
    print(p.relative_to(WORK))


Source copied successfully.

Python files:
code/business_entity_resolution/src/business_entity_resolution/__init__.py
code/business_entity_resolution/src/business_entity_resolution/__main__.py
code/business_entity_resolution/src/business_entity_resolution/blocking/__init__.py
code/business_entity_resolution/src/business_entity_resolution/blocking/candidate_generator.py
code/business_entity_resolution/src/business_entity_resolution/blocking/exact.py
code/business_entity_resolution/src/business_entity_resolution/blocking/recall_eval.py
code/business_entity_resolution/src/business_entity_resolution/blocking/tfidf_blocker.py
code/business_entity_resolution/src/business_entity_resolution/blocking/token.py
code/business_entity_resolution/src/business_entity_resolution/cli.py
code/business_entity_resolution/src/business_entity_resolution/config.py
code/business_entity_resolution/src/business_entity_resolution/data/__init__.py
code/business_entity_resolution/src/business_entity_resolution/data

In [17]:
from pathlib import Path
import sys
import os

REPO_ROOT = Path("/kaggle/working/ml_amazon")
DATASET_ROOT = Path(
    "/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data"
)

# Create the repository's expected dataset directory
dataset_dir = REPO_ROOT / "dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)

# Link Kaggle's read-only dataset into the repository layout
for split in ["train", "test"]:
    target = dataset_dir / split
    source = DATASET_ROOT / split

    if target.exists() or target.is_symlink():
        if target.is_symlink():
            target.unlink()
        elif target.is_dir():
            import shutil
            shutil.rmtree(target)

    if source.exists():
        target.symlink_to(source, target_is_directory=True)
        print(f"Linked {split}: {target} -> {source}")
    else:
        print(f"WARNING: {split} does not exist in Kaggle dataset")

# Make the src package importable
SRC_ROOT = REPO_ROOT / "code" / "business_entity_resolution" / "src"
sys.path.insert(0, str(SRC_ROOT))

print("\nREPO_ROOT:", REPO_ROOT)
print("SRC_ROOT:", SRC_ROOT)

# Verify package + config
from business_entity_resolution.config import (
    REPO_ROOT as CONFIG_REPO_ROOT,
    TRAIN_DIR,
    TRAIN_FILES,
    TRAIN_GROUND_TRUTH,
)

print("\nConfig REPO_ROOT:", CONFIG_REPO_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)

print("\nTraining files:")
for name, path in TRAIN_FILES.items():
    print(f"{name}: {path} | exists={path.exists()}")

print(
    "GROUND TRUTH:",
    TRAIN_GROUND_TRUTH,
    "| exists=",
    TRAIN_GROUND_TRUTH.exists(),
)

Linked train: /kaggle/working/ml_amazon/dataset/train -> /kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data/train

REPO_ROOT: /kaggle/working/ml_amazon
SRC_ROOT: /kaggle/working/ml_amazon/code/business_entity_resolution/src

Config REPO_ROOT: /kaggle/working/ml_amazon
TRAIN_DIR: /kaggle/working/ml_amazon/dataset/train

Training files:
S1: /kaggle/working/ml_amazon/dataset/train/train_source1.tsv | exists=True
S2: /kaggle/working/ml_amazon/dataset/train/train_source2.tsv | exists=True
S3: /kaggle/working/ml_amazon/dataset/train/train_source3.tsv | exists=True
GROUND TRUTH: /kaggle/working/ml_amazon/dataset/train/train_ground_truth.tsv | exists= True


In [18]:
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ml_amazon")
SRC_ROOT = REPO_ROOT / "code" / "business_entity_resolution" / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Import every major pipeline component.
from business_entity_resolution.config import DEFAULT_CONFIG
from business_entity_resolution.data.loader import *
from business_entity_resolution.normalization.names import *
from business_entity_resolution.normalization.addresses import *
from business_entity_resolution.blocking.candidate_generator import *
from business_entity_resolution.features.build_feature_matrix import *
from business_entity_resolution.models.train_lgbm import *
from business_entity_resolution.models.threshold import *
from business_entity_resolution.pipeline.train_pipeline import *

print("✅ Pipeline imports successful")
print("Random seed:", DEFAULT_CONFIG.random_seed)
print("Token top-k:", DEFAULT_CONFIG.blocking.token_top_k_per_s1)
print("Max candidates/S1:", DEFAULT_CONFIG.blocking.max_candidates_per_s1)
print("LightGBM estimators:", DEFAULT_CONFIG.model.n_estimators)

✅ Pipeline imports successful
Random seed: 42
Token top-k: 20
Max candidates/S1: 25
LightGBM estimators: 500


In [20]:
import inspect
from business_entity_resolution.pipeline import train_pipeline

print("=== FUNCTIONS ===")
for name, obj in inspect.getmembers(train_pipeline, inspect.isfunction):
    print(name, inspect.signature(obj))

print("\n=== MODULE SOURCE ===")
print(inspect.getsource(train_pipeline))

=== FUNCTIONS ===
_rss_mb () -> 'float'
blocking_recall (candidates: 'pd.DataFrame', ground_truth_long: 'pd.DataFrame', breakdown_cols: 'dict[str, pd.Series] | None' = None) -> 'dict'
build_feature_matrix (candidates: 'pd.DataFrame', s1_normalized: 'pd.DataFrame', others_normalized: 'pd.DataFrame', cfg: 'FeatureConfig', chunk_size: 'int' = 4000000) -> 'tuple[pd.DataFrame, pd.DataFrame]'
feature_importance (model: 'lgb.LGBMClassifier', feature_names: 'list[str]') -> 'pd.DataFrame'
full_validation_report (predictions: 'dict[str, set]', ground_truth: 'dict[str, set]', beta: 'float' = 0.5) -> 'dict'
label_candidates (candidates: 'pd.DataFrame', ground_truth_long: 'pd.DataFrame') -> 'pd.DataFrame'
load_normalize_block (s1n: 'pd.DataFrame', source_path: 'Path', source_label: 'str', cfg: 'BlockingConfig', nrows: 'int | None' = None) -> 'tuple[pd.DataFrame, pd.DataFrame]'
normalize_source (df: 'pd.DataFrame') -> 'pd.DataFrame'
predict_matches (scored: 'pd.DataFrame', tau: 'float', apply_revers

In [21]:
!python /kaggle/working/ml_amazon/code/business_entity_resolution/exp3_script.py

2026-09-27 06:16:16,145 | INFO | loading S1 + ground truth (nrows=10000)
/kaggle/working/ml_amazon/code/business_entity_resolution/src/business_entity_resolution/pipeline/sampling.py:55: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  strata.groupby(["country", "match_count"], group_keys=False)
/kaggle/working/ml_amazon/code/business_entity_resolution/src/business_entity_resolution/pipeline/sampling.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=seed))
/kaggle

Starting small pipeline smoke test...


2026-09-27 06:16:16,365 | INFO | training on 1999 S1 entities (stratified sample, seed=42)
2026-09-27 06:16:16,596 | INFO | blocking (S2 and S3 processed sequentially to bound peak memory)
2026-09-27 06:16:18,036 | INFO | S2: 49889 candidate rows referencing 8662 unique entities (kept in memory for features)
2026-09-27 06:16:19,501 | INFO | S3: 50717 candidate rows referencing 8698 unique entities (kept in memory for features)
2026-09-27 06:16:19,780 | INFO | blocking recall: 0.0000 (0/35 true pairs) | avg candidates/s1=23.7
2026-09-27 06:16:19,781 | INFO | labeling
2026-09-27 06:16:19,803 | INFO | labeled pairs: positive=0 negative=47234 (ratio 1:47234.0)
2026-09-27 06:16:19,951 | INFO | feature engineering
2026-09-27 06:16:22,457 | INFO | feature engineering RSS: before=602MB after=650MB delta=47MB
2026-09-27 06:16:22,458 | INFO | train/val split (fold split over the FULL S1 id space, not just ids with candidates)
2026-09-27 06:16:22,470 | INFO | training LightGBM on 37688 rows (0 po


SMOKE TEST COMPLETE
Runtime: 6.5 sec
S1 sampled: 1999
Candidate pairs: 47234
Positive pairs: 0
Negative pairs: 47234
Blocking recall: 0.0
Best threshold: 0.05
Validation F0.5: 0.9975
